In [42]:
import pandas as pd
timetable = pd.read_csv('syu_timetable_result.csv')
bucket = pd.read_csv('syu_basket_result_fixed.csv')

In [43]:
timetable

,강의명,강좌번호,시간,전공
0,4차산업과 인공지능,1096,목2~4,공통(교양)
1,간호학개론,373,화4~5,간호학과
2,간호학개론,374,화6~7,간호학과
3,감성 디자인을 위한 AI와 일러스트,966,수1~3,공통(교양)
4,건강과운동,878,화1~3,공통(교양)
...,...,...,...,...
1568,(KLEC)대학한국어 말하기Ⅱ,1881,?,공통(교양)
1569,(KLEC)대학한국어 쓰기Ⅰ,962,?,공통(교양)
1570,(KLEC)대학한국어 쓰기Ⅱ,1882,?,공통(교양)
1571,(KLEC)대학한국어 읽기Ⅰ,1043,?,공통(교양)


In [44]:
bucket

,강좌번호,장바구니 인원,수강정원
0,2,33,50
1,3,8,75
2,4,13,35
3,5,63,35
4,6,19,40
...,...,...,...
1401,1837,59,15
1402,1838,26,15
1403,1839,1,40
1404,1840,12,40


In [45]:
location_map = {
    "SW보건빅데이터": "에스라관",
    "SW보건빅데이터 / 보건빅데이터": "에스라관",
    "SW중독재활": "에스라관",
    "SW중독심리": "에스라관",
    "간호학과": "다니엘관",
    "건축학과(4년)": "디자인관",
    "건축학과(5년)": "디자인관",
    "경영학과": "바울관",
    "공통(교양)": "다니엘관",
    "글로벌한국학과": "에스라관",
    "데이터클라우드공학과": "제1실습관",
    "동물자원과학과": "제2과학관",
    "물리치료학과": "다니엘관",
    "미래융합자유전공학부": "다니엘관",
    "바이오융합공학과": "다니엘관",
    "보건관리학과": "에스라관",
    "사회복지학과": "사무엘관",
    "상담심리학과": "국제교육관",
    "식품영양학과": "다니엘관",
    "신학과": "신학관",
    "아트앤디자인학과": "아트관",
    "약학과": "제3과학관",
    "약학과 (6년제)": "제3과학관",
    "영어영문학과": "사무엘관",
    "유아교육과": "제1실습관",
    "음악학과": "음악관",
    "인공지능융합학부": "제1실습관",
    "체육학과": "체육관",
    "컴퓨터공학부": "제1실습관",
    "항공관광외국어학부": "바울관",
    "화학생명과학과": "제2과학관",
    "환경디자인원예학과": "제2과학관",
}

In [46]:
timetable["강의실"] = timetable["전공"].map(location_map).fillna("")
timetable = timetable[timetable["시간"] != "?"]
timetable

,강의명,강좌번호,시간,전공,강의실
0,4차산업과 인공지능,1096,목2~4,공통(교양),다니엘관
1,간호학개론,373,화4~5,간호학과,다니엘관
2,간호학개론,374,화6~7,간호학과,다니엘관
3,감성 디자인을 위한 AI와 일러스트,966,수1~3,공통(교양),다니엘관
4,건강과운동,878,화1~3,공통(교양),다니엘관
...,...,...,...,...,...
1562,필수실무실습Ⅱ,88,일3~5,약학과,제3과학관
1563,필수실무실습Ⅲ,90,일6~7,약학과,제3과학관
1564,필수실무실습Ⅳ,87,일8~9,약학과,제3과학관
1565,필수실무실습Ⅴ,89,일10~11,약학과,제3과학관


In [47]:
def parse_time(time_str):
    results = []
    for part in str(time_str).split(","):
        # "월5~7" 같은 경우
        match = re.match(r"([가-힣]+)(\d+)~(\d+)", part)
        if match:
            results.append((match.group(1), int(match.group(2)), int(match.group(3))))
        else:
            # "월9" 같은 경우 (끝 교시 없음 → 시작=끝)
            match2 = re.match(r"([가-힣]+)(\d+)", part)
            if match2:
                num = int(match2.group(2))
                results.append((match2.group(1), num, num))
    return results if results else [(None, None, None)]

# 첫 번째 시간대만 추출해서 새로운 열로 추가 (assign 사용)
timetable = timetable.assign(
    요일 = timetable["시간"].apply(lambda x: parse_time(x)[0][0]),
    수업시작 = timetable["시간"].apply(lambda x: parse_time(x)[0][1]),
    수업끝 = timetable["시간"].apply(lambda x: parse_time(x)[0][2])
)
timetable

,강의명,강좌번호,시간,전공,강의실,요일,수업시작,수업끝
0,4차산업과 인공지능,1096,목2~4,공통(교양),다니엘관,목,2,4
1,간호학개론,373,화4~5,간호학과,다니엘관,화,4,5
2,간호학개론,374,화6~7,간호학과,다니엘관,화,6,7
3,감성 디자인을 위한 AI와 일러스트,966,수1~3,공통(교양),다니엘관,수,1,3
4,건강과운동,878,화1~3,공통(교양),다니엘관,화,1,3
...,...,...,...,...,...,...,...,...
1562,필수실무실습Ⅱ,88,일3~5,약학과,제3과학관,일,3,5
1563,필수실무실습Ⅲ,90,일6~7,약학과,제3과학관,일,6,7
1564,필수실무실습Ⅳ,87,일8~9,약학과,제3과학관,일,8,9
1565,필수실무실습Ⅴ,89,일10~11,약학과,제3과학관,일,10,11


In [48]:
merged = pd.merge(
    timetable,
    bucket,
    on="강좌번호",   # 기준 열
    how="left"      # timetable 기준으로 bucket 붙이기
)
merged

,강의명,강좌번호,시간,전공,강의실,요일,수업시작,수업끝,장바구니 인원,수강정원
0,4차산업과 인공지능,1096,목2~4,공통(교양),다니엘관,목,2,4,28.0,40.0
1,간호학개론,373,화4~5,간호학과,다니엘관,화,4,5,3.0,45.0
2,간호학개론,374,화6~7,간호학과,다니엘관,화,6,7,0.0,45.0
3,감성 디자인을 위한 AI와 일러스트,966,수1~3,공통(교양),다니엘관,수,1,3,52.0,32.0
4,건강과운동,878,화1~3,공통(교양),다니엘관,화,1,3,75.0,24.0
...,...,...,...,...,...,...,...,...,...,...
1498,필수실무실습Ⅱ,88,일3~5,약학과,제3과학관,일,3,5,0.0,40.0
1499,필수실무실습Ⅲ,90,일6~7,약학과,제3과학관,일,6,7,0.0,40.0
1500,필수실무실습Ⅳ,87,일8~9,약학과,제3과학관,일,8,9,0.0,40.0
1501,필수실무실습Ⅴ,89,일10~11,약학과,제3과학관,일,10,11,0.0,40.0


In [49]:
merged["실 수강생"] = merged.apply(
    lambda row: min(row["장바구니 인원"], row["수강정원"]) 
    if pd.notnull(row["장바구니 인원"]) and pd.notnull(row["수강정원"]) 
    else None,
    axis=1
)
merged

,강의명,강좌번호,시간,전공,강의실,요일,수업시작,수업끝,장바구니 인원,수강정원,실 수강생
0,4차산업과 인공지능,1096,목2~4,공통(교양),다니엘관,목,2,4,28.0,40.0,28.0
1,간호학개론,373,화4~5,간호학과,다니엘관,화,4,5,3.0,45.0,3.0
2,간호학개론,374,화6~7,간호학과,다니엘관,화,6,7,0.0,45.0,0.0
3,감성 디자인을 위한 AI와 일러스트,966,수1~3,공통(교양),다니엘관,수,1,3,52.0,32.0,32.0
4,건강과운동,878,화1~3,공통(교양),다니엘관,화,1,3,75.0,24.0,24.0
...,...,...,...,...,...,...,...,...,...,...,...
1498,필수실무실습Ⅱ,88,일3~5,약학과,제3과학관,일,3,5,0.0,40.0,0.0
1499,필수실무실습Ⅲ,90,일6~7,약학과,제3과학관,일,6,7,0.0,40.0,0.0
1500,필수실무실습Ⅳ,87,일8~9,약학과,제3과학관,일,8,9,0.0,40.0,0.0
1501,필수실무실습Ⅴ,89,일10~11,약학과,제3과학관,일,10,11,0.0,40.0,0.0


In [50]:
merged = merged.drop(
    columns=["강의명", "강좌번호", "시간", "전공", "장바구니 인원", "수강정원"]
)

In [51]:
merged

,강의실,요일,수업시작,수업끝,실 수강생
0,다니엘관,목,2,4,28.0
1,다니엘관,화,4,5,3.0
2,다니엘관,화,6,7,0.0
3,다니엘관,수,1,3,32.0
4,다니엘관,화,1,3,24.0
...,...,...,...,...,...
1498,제3과학관,일,3,5,0.0
1499,제3과학관,일,6,7,0.0
1500,제3과학관,일,8,9,0.0
1501,제3과학관,일,10,11,0.0


In [52]:
# 조건: 강의실=다니엘관, 요일=월, 수업시작=1
filtered = merged[
    (merged["강의실"] == "다니엘관") &
    (merged["요일"] == "월") &
    (merged["수업시작"] == 1)
]

# 실 수강생 합계
total_students = filtered["실 수강생"].sum()

print("실 수강생 합:", total_students)

실 수강생 합: 250.0


In [68]:
import pandas as pd

# 1. 기초 설정
rooms = merged['강의실'].unique()
days_order = ['월', '화', '수', '목', '금', '토', '일'] # 우리가 원하는 순서 정의
hours = range(1, 16)

# 2. 모든 조합 생성
all_combos = pd.MultiIndex.from_product([rooms, days_order, hours], names=['강의실', '요일', '시간'])
new_df = pd.DataFrame(index=all_combos).reset_index()

# 3. 매핑 데이터 생성
start_map = merged.groupby(['강의실', '요일', '수업시작'])['실 수강생'].sum().to_dict()
end_map = merged.groupby(['강의실', '요일', '수업끝'])['실 수강생'].sum().to_dict()

def calculate_neighbor_sum(row):
    r, d, t = row['강의실'], row['요일'], row['시간']
    return end_map.get((r, d, t), 0) + start_map.get((r, d, t), 0)

# 4. 합계 계산
new_df['합계'] = new_df.apply(calculate_neighbor_sum, axis=1)

# ---------------------------------------------------------
# 5. [핵심] 요일 정렬 적용 부분
# ---------------------------------------------------------
# '요일' 컬럼을 우리가 정의한 days_order 순서를 가진 '카테고리형'으로 변환합니다.
new_df['요일'] = pd.Categorical(new_df['요일'], categories=days_order, ordered=True)

# 이제 sort_values를 하면 월, 화, 수 순서대로 정렬됩니다.
result_df = new_df.sort_values(['강의실', '요일', '시간'])
# ---------------------------------------------------------

# 6. CSV 저장 및 확인
result_df.to_csv('lecture_summary.csv', index=False, encoding='utf-8-sig')

print("요일 정렬(월~일)이 적용된 파일이 저장되었습니다.")
result_df.head(20)

요일 정렬(월~일)이 적용된 파일이 저장되었습니다.


,강의실,요일,시간,합계
945,국제교육관,월,1,0.0
946,국제교육관,월,2,35.0
947,국제교육관,월,3,0.0
948,국제교육관,월,4,50.0
949,국제교육관,월,5,0.0
950,국제교육관,월,6,55.0
951,국제교육관,월,7,0.0
952,국제교육관,월,8,40.0
953,국제교육관,월,9,0.0
954,국제교육관,월,10,0.0
